In [1]:
import langchain; print(langchain.__version__)

1.4.2


In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# 예시: 블로그 글 하나를 불러옵니다
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

print(f"불러온 문서 수: {len(docs)}")
print(f"글자 수: {len(docs[0].page_content)}")
print(docs[0].page_content[:500])  # 앞부분 500자만 미리보기

/var/folders/8b/tptjhw711rl8y2287cz1dzz00000gn/T/ipykernel_53033/3745805156.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


불러온 문서 수: 1
글자 수: 43047


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # 청크 하나당 최대 글자 수
    chunk_overlap=200,    # 청크끼리 겹치는 부분 (문맥 끊김 방지)
)
splits = text_splitter.split_documents(docs)

print(f"생성된 청크 수: {len(splits)}")
print(f"첫 번째 청크 미리보기:\n{splits[0].page_content[:300]}")

생성된 청크 수: 63
첫 번째 청크 미리보기:
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring


In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
)

print("벡터스토어 생성 완료!")
print(f"저장된 청크 수: {vectorstore._collection.count()}")

벡터스토어 생성 완료!
저장된 청크 수: 63


In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

question = "What is task decomposition?"
retrieved_docs = retriever.invoke(question)

print(f"검색된 청크 수: {len(retrieved_docs)}")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- 청크 {i+1} ---")
    print(doc.page_content[:200])

검색된 청크 수: 4

--- 청크 1 ---
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a s

--- 청크 2 ---
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outl

--- 청크 3 ---
The AI assistant can parse user input to several tasks: [{"task": task, "id", task_id, "dep": dependency_task_ids, "args": {"text": text, "image": URL, "audio": URL, "video": URL}}]. The "dep" field d

--- 청크 4 ---
(3) Task execution: Expert models execute on the specific tasks and log results.
Instruction:

With the input and the inference results, the AI assistant needs to describe the process and results. The


In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_template("""
다음 컨텍스트만 참고해서 질문에 답변하세요. 컨텍스트에 없는 내용이면 모른다고 답하세요.

컨텍스트:
{context}

질문: {question}

답변:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

context = format_docs(retrieved_docs)
chain = prompt | llm

response = chain.invoke({"context": context, "question": question})
print(response.content)

Task decomposition is the process of breaking down a complicated task into smaller, more manageable steps. This allows an agent, such as an AI model, to plan ahead and tackle complex problems more effectively. There are various methods for task decomposition, including using simple prompting techniques like asking for the "Steps for XYZ," employing task-specific instructions, or incorporating human inputs. 

Notably, techniques like Chain of Thought (CoT) and Tree of Thoughts (ToT) enhance the decomposition process by encouraging the model to think step by step and explore multiple reasoning possibilities. CoT transforms a large task into smaller parts, while ToT creates a tree structure by generating multiple thoughts for each step. Additionally, another approach called LLM+P utilizes an external classical planner to assist with long-horizon planning, leveraging Planning Domain Definition Language (PDDL) as an intermediate interface.
